In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# 1. VARIABLES

#unit conversions
temp_F = (temp - 273.15) * 9/5 + 32
dew_F = (dew - 273.15) * 9/5 + 32
precip_mm = precip * 1000
wind_mph = wind_gust * 2.237
snow_mm = snow * 1000
times = ds2['valid_time'].values

#Relavtive Humidity Def: 
def calc_rh(T, Td):
    return 100 * np.exp((17.625 * Td)/(243.04 + Td)) / np.exp((17.625 * T)/(243.04 + T))

#Wet Bulb Def: 
def wet_bulb_temp(T, RH):
    return T * np.arctan(0.151977 * (RH + 8.313659)**0.5) + \
           np.arctan(T + RH) - np.arctan(RH - 1.676331) + \
           0.00391838 * RH**1.5 * np.arctan(0.023101 * RH) - 4.686035

def wbgt_calc(T, Tw):
    return 0.7 * Tw + 0.3 * T

# Store results
precip_risk = []
wind_risk = []
wbgt_risk = []
field_risk = []

times = ds2['valid_time'].values #variable for valid times

for t in range(len(times)):

    T = temp_F.isel(valid_time=t) #finding the valid times for each variable for the sake of the hazard
    Td = dew_F.isel(valid_time=t)
    P = precip_mm.isel(valid_time=t)
    S = snow_mm.isel(valid_time=t)
    W = wind_mph.isel(valid_time=t)

    total = T.size #size of the data collection

 
    # WBGT
    
    RH = calc_rh(T, Td)
    Tw = wet_bulb_temp(T, RH)
    WBGT = wbgt_calc(T, Tw)

    wbgt_low25 = (WBGT <= 25).sum() / total * 100 #hazard level for wet bulb
    wbgt_low32 = (WBGT <= 32).sum() / total * 100
#to put on the graph of what level each hazard is
    if wbgt_low25 > 75: 
        wbgt_risk.append(100)
    elif wbgt_low25 >= 40 or wbgt_low32 > 75:
        wbgt_risk.append(60)
    else:
        wbgt_risk.append(20)

   
    # FIELD CONDITIONS
   
    heavy_precip = (P >= 5).sum() / total * 100 #same thing but for precip
    any_precip = (P >= 0.5).sum() / total * 100
    snow_cover = (S >= 2).sum() / total * 100

    freezing = (T <= 32).sum() / total * 100

    if snow_cover > 50 or heavy_precip > 75:
        field_risk.append(100)
    elif any_precip > 40 or (snow_cover > 25 and freezing < 50):
        field_risk.append(60)
    else:
        field_risk.append(20)

   
    # PRECIP

    heavy = (P >= 5).sum() / total * 100
    moderate = (P >= 2).sum() / total * 100

    if heavy > 75:
        precip_risk.append(100)
    elif moderate >= 40:
        precip_risk.append(60)
    else:
        precip_risk.append(20)

    
    # WIND
    
    strong = (W >= 15).sum() / total * 100

    if strong > 75:
        wind_risk.append(100)
    elif strong >= 40:
        wind_risk.append(60)
    else:
        wind_risk.append(20)

# 3. PLOT


fig, ax = plt.subplots(figsize=(12,6))

ax.plot(times, precip_risk, label='Precip Risk', linewidth=2) #plotting each variable by itself, but on the same graph
ax.plot(times, wind_risk, label='Wind Risk', linewidth=2)
ax.plot(times, wbgt_risk, label='WBGT Risk', linewidth=2)
ax.plot(times, field_risk, label='Field Condition Risk', linewidth=2)

# Overall max risk (decision line)
#overall = np.maximum.reduce([precip_risk, wind_risk, wbgt_risk, field_risk])
#ax.plot(times, overall, color='black', linewidth=3, linestyle='--', label='Overall Risk')

# Risk bands
ax.axhspan(75, 100, alpha=0.2)
ax.axhspan(40, 75, alpha=0.1)

ax.set_ylim(0, 100)
ax.set_ylabel('Risk (%)')
ax.set_xlabel('Forecast Time')
ax.set_title(
    f"Game Cancellation Risk Assessment (Multi-Factor)\n"
    f"Inital: {np.datetime_as_string(ds2['time'], unit='h')}\n"
    f"Valid: {np.datetime_as_string(ds2['valid_time'].values[-1], unit='h')}"
)

ax.legend()
plt.xticks(rotation=45)
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Create normalization (0–100%)
norm = Normalize(vmin=0, vmax=100)

# Blue colormap
cmap = cm.Blues

# Create a ScalarMappable for the colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Add background shading using same cmap
ax.axhspan(0, 40, color=cmap(norm(20)), alpha=0.3)
ax.axhspan(40, 75, color=cmap(norm(60)), alpha=0.3)
ax.axhspan(75, 100, color=cmap(norm(90)), alpha=0.3)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Risk Level (%)')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# 1. Select the PA region

lat_min, lat_max = 39.5, 42.5
lon_min, lon_max = -80.5, -74.5


# 2. Extract last forecast step

tp = ds2['tp'].sel(valid_time=ds2['valid_time'].values[-1], method='nearest') #total precip
fg10 = ds2['fg10'].sel(valid_time=ds2['valid_time'].values[-1], method='nearest') #10 m gusts
d2m = ds2['d2m'].sel(valid_time=ds2['valid_time'].values[-1], method='nearest') #dew point
t2m = ds2['t2m'].sel(valid_time=ds2['valid_time'].values[-1], method='nearest') #temp
sf = ds2['sf'].sel(valid_time=ds2['valid_time'].values[-1], method='nearest')       # snowfall

lons = ds2['longitude'].values
lats = ds2['latitude'].values
LON, LAT = np.meshgrid(lons, lats) #proper lon and lat for PA


# 3. Compute risk per grid point

# Example thresholds (same idea as your previous code)
precip_risk_map = np.where(tp > 5, 100, 20)        # precipitation threshold
wind_risk_map = np.where(fg10 > 15, 100, 20)      # strong wind threshold
wbgt_risk_map = np.where((t2m > 30) & (d2m > 20), 100, 20)  # WBGT heat risk
field_risk_map = np.where(sf > 0, 100, 20)        # snow on field

# Overall risk = max of all risks
overall_risk_map = np.maximum.reduce([precip_risk_map, wind_risk_map, wbgt_risk_map, field_risk_map])


# 4. Plot all risk maps

fig, axes = plt.subplots(3, 2, figsize=(15, 18),
                         subplot_kw={'projection': ccrs.PlateCarree()}) #PlateCaree is type of map projection
axes = axes.flatten()

risk_maps = [precip_risk_map, wind_risk_map, wbgt_risk_map, field_risk_map, overall_risk_map]
titles = ['Precipitation Risk', 'Wind Risk', 'WBGT Risk', 'Field Condition Risk', 'Overall Risk']
colors = ['Blues', 'Oranges', 'Reds', 'Purples', 'Greens']

for ax, risk, title, cmap in zip(axes, risk_maps, titles, colors):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    ax.add_feature(cfeature.STATES, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.LAKES, facecolor='lightblue')
    ax.add_feature(cfeature.RIVERS) #all of these add_features are to make the map cooler
    
    c = ax.contourf(LON, LAT, risk, levels=[0,20,40,75,100], cmap=cmap, alpha=0.6) #contourf is type of plot
    cb = plt.colorbar(c, ax=ax, orientation='vertical', pad=0.02)
    cb.set_label('Risk (%)')
    ax.set_title(title)

# Remove last empty subplot (axes[5]) if not used
fig.delaxes(axes[-1])

# ==============================
# 5. Add overall title
# ==============================
time_initial = ds2['time'].values
time_end = ds2['valid_time'].values[-1]

fig.suptitle(
    f"Pennsylvania Game Risk Maps\nForecast Initialized: {np.datetime_as_string(time_initial, unit='h')}\n"
    f"Valid Until: {np.datetime_as_string(time_end, unit='h')}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()